In [ ]:
# DATA PREPROCESSING 

import json
import cv2
import os
import glob
from tqdm.notebook import tqdm
import shutil


ROOT_DIR = "/kaggle/input/olympic-boxing-punch-classification-video-dataset/Olympic Boxing Punch Classification Video Dataset"
OUTPUT_DIR = "/kaggle/working/processed_dataset"

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR)

def get_universal_label(raw_label):
    if not raw_label: return None
    txt = str(raw_label).lower()
    
    is_head = "lowa" in txt or "head" in txt or "głowa" in txt
    is_body = "orpus" in txt or "body" in txt or "korpus" in txt
    is_left = "lew" in txt or "left" in txt
    is_right = "raw" in txt or "right" in txt or "praw" in txt
    
    if "miss" in txt or "chybienie" in txt or "block" in txt or "blok" in txt:
        return None

    if is_head and is_left: return "Head_Left"
    if is_head and is_right: return "Head_Right"
    if is_body and is_left: return "Body_Left"
    if is_body and is_right: return "Body_Right"
    return None

def extract_clips(video_path, json_path, task_name):
    try:
        with open(json_path, 'r', encoding='utf-8-sig') as f:
            data = json.load(f)
    except Exception as e:
        print(f" Error reading JSON file {task_name}: {e}")
        return 0
        
    if isinstance(data, list):
        if len(data) > 0:
            data = data[0]
        else:
            return 0

    if 'tracks' not in data:
        print(f" 'tracks' key missing in {task_name}")
        return 0

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        
        return 0

    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    action_count = 0
    
    for track in data['tracks']:
        raw_label = track.get('label')
        label_en = get_universal_label(raw_label)
        
        if label_en is None: continue
            
        class_dir = os.path.join(OUTPUT_DIR, label_en)
        os.makedirs(class_dir, exist_ok=True)
        
        frames = [shape['frame'] for shape in track['shapes']]
        if not frames: continue
            
        start_frame = min(frames)
        end_frame = max(frames)
        
        pad = 8 
        start_frame = max(0, start_frame - pad)
        end_frame = min(total_frames, end_frame + pad)
        
        if end_frame - start_frame < 5: continue

        out_filename = f"{label_en}_{task_name}_{action_count}.mp4"
        out_path = os.path.join(class_dir, out_filename)
        
        if os.path.exists(out_path):
            action_count += 1
            continue

        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(out_path, fourcc, fps, (width, height))
        
        for _ in range(end_frame - start_frame):
            ret, frame = cap.read()
            if not ret: break
            out.write(frame)
            
        out.release()
        action_count += 1
        
    cap.release()
    return action_count

task_folders = glob.glob(os.path.join(ROOT_DIR, "task_*"))

total_clips = 0
for folder in tqdm(task_folders, desc="Processing"):
    task_name = os.path.basename(folder)
    
    video_files = glob.glob(os.path.join(folder, "**", "*.mp4"), recursive=True) + \
                  glob.glob(os.path.join(folder, "**", "*.MOV"), recursive=True)
    
    json_files = glob.glob(os.path.join(folder, "**", "annotations.json"), recursive=True)
    
    if not json_files:
        json_files = glob.glob(os.path.join(folder, "**", "*.json"), recursive=True)
    
    if video_files and json_files:
        extracted = extract_clips(video_files[0], json_files[0], task_name)
        total_clips += extracted

print(f"\n Extraction Complete!")
print(f"Total Clean Clips: {total_clips}")
print(f"Location: {OUTPUT_DIR}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import cv2
import numpy as np
import os
import glob
import math
import gc
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
from collections import Counter
from torch.cuda.amp import autocast, GradScaler

try:
    del model, optimizer, criterion, train_loader, sampler, scheduler
    gc.collect()
    torch.cuda.empty_cache()
except:
    pass


DATA_ROOT = "/kaggle/input/dataset1/processed_dataset" 
BATCH_SIZE = 32        
NUM_EPOCHS = 25          
LEARNING_RATE = 0.0001   
SEQUENCE_LENGTH = 16
IMG_SIZE = 224
CLASSES = ["Head_Left", "Head_Right", "Body_Left", "Body_Right"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.device_count() > 1:
    

class AttentionModule(nn.Module):
    def __init__(self, embed_size, heads):
        super(AttentionModule, self).__init__()
        self.embed_size = embed_size
        self.heads = heads
        self.head_dim = embed_size // heads
        assert (self.head_dim * heads == embed_size), "Embed size needs to be divisible by heads"

        self.values = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.keys = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.queries = nn.Linear(self.head_dim, self.head_dim, bias=False)
        self.fc_out = nn.Linear(heads * self.head_dim, embed_size)

    def forward(self, values, keys, query, mask=None):
        N = query.shape[0]
        value_len, key_len, query_len = values.shape[1], keys.shape[1], query.shape[1]

        values = values.reshape(N, value_len, self.heads, self.head_dim)
        keys = keys.reshape(N, key_len, self.heads, self.head_dim)
        queries = query.reshape(N, query_len, self.heads, self.head_dim)

        values = self.values(values)
        keys = self.keys(keys)
        queries = self.queries(queries)

        energy = torch.einsum("nqhd,nkhd->nhqk", [queries, keys])
        if mask is not None:
            energy = energy.masked_fill(mask == 0, float("-1e20"))

        attention = torch.softmax(energy / (self.embed_size ** (1/2)), dim=3)
        out = torch.einsum("nhql,nlhd->nqhd", [attention, values]).reshape(
            N, query_len, self.heads * self.head_dim
        )
        return self.fc_out(out)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=50):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return x + self.pe[:, :x.size(1)]

class ActionTransformer(nn.Module):
    def __init__(self, num_classes=4):
        super(ActionTransformer, self).__init__()
    
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.feature_dim = 256
        self.pos_encoder = PositionalEncoding(self.feature_dim)
       
        self.custom_attention = AttentionModule(embed_size=256, heads=4)
        self.norm = nn.LayerNorm(256)

        self.fc = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.5), nn.Linear(128, num_classes)
        )

    def forward(self, x):
        b, c, t, h, w = x.size()
        x = x.permute(0, 2, 1, 3, 4).reshape(b * t, c, h, w)
        x = self.cnn(x).flatten(1).view(b, t, -1)
        x = self.pos_encoder(x)
        
        attention_out = self.custom_attention(values=x, keys=x, query=x)
        x = x + attention_out 
        x = self.norm(x)
        
        x = x.mean(dim=1)            
        return self.fc(x)

class BoxingDataset(Dataset):
    def __init__(self, video_paths, labels, transform=None):
        self.video_paths = video_paths
        self.labels = labels
        self.transform = transform
        self.seq_len = SEQUENCE_LENGTH
    def __len__(self): return len(self.video_paths)
    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]
        cap = cv2.VideoCapture(video_path)
        frames = []
        while len(frames) < self.seq_len:
            ret, frame = cap.read()
            if not ret: break
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            if self.transform: frame = self.transform(frame)
            frames.append(frame)
        cap.release()
        if len(frames) == 0: return torch.zeros(3, self.seq_len, IMG_SIZE, IMG_SIZE), label
        while len(frames) < self.seq_len: frames.append(frames[-1])
        video_tensor = torch.stack(frames).permute(1, 0, 2, 3) 
        return video_tensor, label

all_paths, all_labels = [], []
label_map = {cls: i for i, cls in enumerate(CLASSES)}

for cls in CLASSES:
    cls_folder = os.path.join(DATA_ROOT, cls)
    vids = glob.glob(os.path.join(cls_folder, "*.mp4"))
    all_paths.extend(vids)
    all_labels.extend([label_map[cls]] * len(vids))
    print(f"   -> {cls}: {len(vids)} videos found")

train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels, test_size=0.2, random_state=42, shuffle=True, stratify=all_labels
)

class_counts = Counter(train_labels)
class_weights = {cls: 1.0/count for cls, count in class_counts.items()}
sample_weights = [class_weights[lbl] for lbl in train_labels]
sampler = WeightedRandomSampler(torch.DoubleTensor(sample_weights), len(sample_weights))

train_tf = transforms.Compose([
    transforms.ToPILImage(), transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2), 
    transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
val_tf = transforms.Compose([
    transforms.ToPILImage(), transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(), transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_loader = DataLoader(BoxingDataset(train_paths, train_labels, train_tf), 
                          batch_size=BATCH_SIZE, sampler=sampler, num_workers=4, pin_memory=True)
val_loader = DataLoader(BoxingDataset(val_paths, val_labels, val_tf), 
                        batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

model = ActionTransformer(num_classes=len(CLASSES)).to(DEVICE)

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

scaler = GradScaler()  
best_acc = 0.0
SAVE_NAME = "best_model.pth" 


for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    for vid, lbl in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
        vid, lbl = vid.to(DEVICE), lbl.to(DEVICE)
        optimizer.zero_grad()
        
        with autocast(): 
            outputs = model(vid)
            loss = criterion(outputs, lbl)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item()
        
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for vid, lbl in val_loader:
            vid, lbl = vid.to(DEVICE), lbl.to(DEVICE)
            outputs = model(vid)
            _, pred = torch.max(outputs, 1)
            total += lbl.size(0)
            correct += (pred == lbl).sum().item()
    
    acc = 100 * correct / total
    avg_loss = running_loss / len(train_loader)
    
    scheduler.step(acc)
    
    save_msg = ""
    if acc > best_acc:
        best_acc = acc
        if isinstance(model, nn.DataParallel):
            torch.save(model.module.state_dict(), SAVE_NAME)
        else:
            torch.save(model.state_dict(), SAVE_NAME)
        save_msg = f"SAVED {SAVE_NAME}!"
        
    print(f"Epoch {epoch+1}: Loss = {avg_loss:.4f} | Val Acc = {acc:.2f}% {save_msg}")

print(f" Best Accuracy: {best_acc:.2f}%")